# Detección de Fraude en el Consumo de Agua

**Proyecto Invictus**

---

## 1. Introducción

El presente cuaderno de trabajo constituye el entregable central del proyecto Invictus. Nuestro objetivo es desarrollar un sistema avanzado para la **detección de anomalías y posibles fraudes en el consumo de agua** en la región de Valencia. 

Para ello, hemos construido un pipeline de datos completo que abarca desde la ingesta y preprocesamiento de múltiples fuentes de datos hasta el entrenamiento de un modelo de `Deep Learning` (Autoencoder) no supervisado, capaz de identificar patrones de consumo anómalos que se desvían de la norma.

La metodología implementada no solo se centra en la precisión del modelo, sino también en la **interpretabilidad y la capacidad de acción**. El sistema final no solo señala una anomalía, sino que busca proporcionar contexto sobre por qué un consumo es considerado atípico.

Además de los datos de consumo, el análisis se enriquecerá con **fuentes de datos externas**, como datos meteorológicos de AEMET, datos socioeconómicos y demográficos del INE, y datos de turismo de GVA. En futuras iteraciones, se planea la inclusión de un dataset de destinos turísticos para mejorar la granularidad de nuestro análisis en zonas de alta estacionalidad.

## 2. Configuración del Entorno

En esta sección, importamos todas las librerías y módulos necesarios para la ejecución del análisis. Hemos encapsulado la lógica de preprocesamiento y modelado en módulos (`.py`) dentro de la carpeta `src/` para mantener este notebook limpio, legible y enfocado en los resultados.

In [ ]:
# Importaciones de librerías estándar
import pandas as pd
import numpy as np
import plotly.express as px
import seaborn as sns
import matplotlib.pyplot as plt

# Importar y configurar el logger
from src.config import get_logger
logger = get_logger(__name__)

# Configuraciones de visualización
sns.set(style="whitegrid")
px.defaults.template = "plotly_dark"

# Importaciones de nuestros módulos
from src.water2fraud.features.preprocessor import WaterPreprocessor
from src.water2fraud.models import trainer, dataset, autoencoder
from src import utils

## 3. Carga y Preprocesamiento de Datos

El primer paso es cargar y procesar los datos de consumo. Nuestro módulo `preprocessor` se encarga de unificar las diferentes fuentes, limpiar los datos, realizar la ingeniería de características y prepararlos para el modelo.

In [ ]:
# Importamos la clase Paths para acceder a las rutas de los ficheros
from src.config import Paths

# --- 1. Carga del Dataset Inicial ---
# El punto de partida de nuestro análisis es el fichero de consumos de AMAEM.
logger.info(f"Cargando dataset crudo desde: {Paths.RAW_CSV_AMAEM}")
raw_df = pd.read_csv(Paths.RAW_CSV_AMAEM)
logger.info(f"Dataset crudo cargado. Dimensiones: {raw_df.shape}")

# --- 2. Ejecución del Pipeline de Preprocesamiento ---
# Nuestro 'WaterPreprocessor' se encarga de todo el trabajo pesado:
# - Limpieza de datos de agua (AMAEM)
# - Enriquecimiento con datos de turismo (INE, GVA)
# - Enriquecimiento con datos climáticos (AEMET) y de satélite (Sentinel)
# - Ingeniería de características (cálculo de turismo no declarado)
# - Escalado de variables para el modelo
logger.info("Iniciando el pipeline de preprocesamiento...")
df_processed, scalers = WaterPreprocessor.process_raw_data(raw_df)
logger.info("Pipeline de preprocesamiento finalizado.")

# --- 3. Verificación del Resultado ---
# Mostramos la cabecera y la información del dataframe procesado y escalado.
print("--- Dataframe Procesado y Escalado ---")
print(df_processed.info())
display(df_processed.head())

# Mostramos también las columnas que se han utilizado finalmente.
final_features = [col for col in WaterPreprocessor.FEATURES if col in df_processed.columns]
print("\n--- Features finales para el modelo ---")
print(final_features)

## 4. Análisis Exploratorio de Datos (EDA)

Antes de modelar, es crucial entender nuestros datos. Realizaremos un análisis exploratorio para visualizar las distribuciones, identificar tendencias y estacionalidades en el consumo de agua.

In [ ]:
# Importamos la clase DatasetKeys para acceder a las columnas de los datos
from src.config import DatasetKeys

# --- 1. Carga de los datos procesados (pero no escalados) para el EDA ---
# Para el análisis exploratorio, es más intuitivo trabajar con las magnitudes originales.
logger.info(f"Cargando dataset procesado no escalado desde: {Paths.PROC_CSV_AMAEM_NOT_SCALED}")
df_eda = pd.read_csv(Paths.PROC_CSV_AMAEM_NOT_SCALED)

# Aseguramos que la fecha sea un objeto datetime para poder trabajar con series temporales
df_eda[DatasetKeys.FECHA] = pd.to_datetime(df_eda[DatasetKeys.FECHA])

logger.info("Dataset para EDA cargado y listo.")
display(df_eda.head())


# --- 2. Visualización de la Serie Temporal de Consumo Agregado ---
# Agrupamos por fecha para ver la tendencia general del consumo de agua en la ciudad.
df_temporal = df_eda.groupby(DatasetKeys.FECHA)[DatasetKeys.CONSUMO].sum().reset_index()

fig = px.line(df_temporal, 
              x=DatasetKeys.FECHA, 
              y=DatasetKeys.CONSUMO,
              title='Consumo Total de Agua a lo Largo del Tiempo',
              labels={DatasetKeys.CONSUMO: 'Consumo Total (m³)'})
fig.show()


# --- 3. Distribución del Consumo por Tipo de Uso ---
# Comparamos cómo se distribuye el consumo entre los diferentes tipos de uso (doméstico, industrial, etc.)
fig = px.box(df_eda,
             x=DatasetKeys.USO,
             y=DatasetKeys.CONSUMO,
             title='Distribución del Consumo por Tipo de Uso',
             labels={DatasetKeys.USO: 'Tipo de Uso', DatasetKeys.CONSUMO: 'Consumo (m³)'},
             notched=True) # 'notched' para visualizar mejor las diferencias en medianas
fig.show()

## 5. Entrenamiento del Modelo Autoencoder

El corazón de nuestro sistema de detección de fraude es un Autoencoder. Este modelo de red neuronal se entrena para reconstruir los datos de consumo "normales". Las lecturas que el modelo no puede reconstruir con precisión (alto error de reconstrucción) son señaladas como anomalías.

In [ ]:
# --- 1. Definición de Hiperparámetros ---
# Centralizamos aquí todos los parámetros para el entrenamiento del Autoencoder.
SEQ_LEN = 12          # Usamos secuencias de 12 meses (1 año) para capturar patrones anuales
BATCH_SIZE = 64       # Tamaño del lote para el entrenamiento
EPOCHS = 150          # Número máximo de épocas
LEARNING_RATE = 1e-3  # Tasa de aprendizaje
HIDDEN_DIM = 64       # Dimensión de la primera capa oculta del LSTM
LATENT_DIM = 16       # Dimensión del cuello de botella (espacio latente)
PATIENCE = 20         # Paciencia para el Early Stopping

# --- 2. Creación de Secuencias Temporales ---
# Transformamos el dataframe 2D en secuencias 3D (samples, timesteps, features)
logger.info(f"Creando secuencias de {SEQ_LEN} meses...")
X_sequences, metadata_df, feature_cols = WaterPreprocessor.create_sequences(df_processed, sequence_length=SEQ_LEN)
logger.info("Secuencias creadas correctamente.")

# --- 3. Creación del DataLoader ---
# Preparamos los datos para ser consumidos por PyTorch en lotes.
dataloader = dataset.get_dataloader(X_sequences, batch_size=BATCH_SIZE, shuffle=True)
num_features = X_sequences.shape[2]
logger.info(f"DataLoader listo. Número de features: {num_features}")

# --- 4. Construcción del Modelo Autoencoder ---
# Instanciamos nuestro modelo LSTMAutoencoder con los parámetros definidos.
logger.info("Construyendo el modelo LSTMAutoencoder...")
model = autoencoder.LSTMAutoencoder(
    num_features=num_features,
    hidden_dim=HIDDEN_DIM,
    latent_dim=LATENT_DIM,
    seq_len=SEQ_LEN
)

# --- 5. Entrenamiento del Modelo ---
# Lanzamos el bucle de entrenamiento. El 'trainer' se encarga del early stopping.
logger.info("Iniciando el entrenamiento del modelo...")
model, history = trainer.train_autoencoder(
    model,
    dataloader,
    epochs=EPOCHS,
    lr=LEARNING_RATE,
    patience=PATIENCE,
    device='cpu' # Usamos CPU para este ejemplo
)
logger.info("Entrenamiento finalizado.")

# --- 6. Visualización de la Curva de Aprendizaje ---
# Es fundamental comprobar que la pérdida ha descendido y se ha estabilizado.
print("\n--- Curva de Pérdida del Entrenamiento ---")
trainer.plot_training_history(history)

## 6. Detección de Anomalías y Análisis de Resultados

Una vez entrenado el modelo, lo utilizamos para obtener el error de reconstrucción de cada punto de datos. Definimos un umbral para clasificar los consumos como normales o anómalos y analizamos los resultados.

In [ ]:
from src.config import AIConstants

# --- 1. Detección de Anomalías ---
# Usamos el modelo entrenado para calcular el error de reconstrucción para cada secuencia.
logger.info("Calculando errores de reconstrucción para detectar anomalías...")
results_df, anomaly_threshold = trainer.detect_ae_anomalies(
    model,
    X_sequences,
    metadata_df,
    feature_names=feature_cols,
    device='cpu'
)
logger.info(f"Detección finalizada. Umbral de anomalía (percentil {AIConstants.AE_ANOMALIES_PERCENTILE}): {anomaly_threshold:.4f}")

# --- 2. Análisis de las Anomalías Principales ---
# Ordenamos los resultados por el error para ver los puntos más anómalos.
anomalies_df = results_df[results_df[DatasetKeys.IS_AE_ANOMALY]].sort_values(
    by=DatasetKeys.RECONSTRUCTION_ERROR, ascending=False
)

print(f"\nSe han detectado {len(anomalies_df)} anomalías.")
print("--- Top 5 Anomalías Detectadas ---")
display(anomalies_df.head())


# --- 3. Visualización de la Distribución de Errores ---
# Un histograma nos permite ver cómo se separan los datos normales de los anómalos.
fig = px.histogram(results_df, 
                   x=DatasetKeys.RECONSTRUCTION_ERROR,
                   title='Distribución del Error de Reconstrucción',
                   labels={DatasetKeys.RECONSTRUCTION_ERROR: 'Error de Reconstrucción (MAE)'},
                   marginal='box')
fig.add_vline(x=anomaly_threshold, line_dash="dash", line_color="red", 
              annotation_text=f"Umbral ({AIConstants.AE_ANOMALIES_PERCENTILE}%)")
fig.show()


# --- 4. Análisis de una Anomalía Específica ---
# Seleccionamos la anomalía con el mayor error y visualizamos su reconstrucción.
if not anomalies_df.empty:
    top_anomaly_idx = anomalies_df.index[0]
    
    # Buscamos el índice del feature 'consumo_ratio'
    try:
        consumo_feature_idx = feature_cols.index(DatasetKeys.CONSUMO_RATIO)
        
        print(f"\n--- Análisis de la Anomalía Principal (Índice: {top_anomaly_idx}) ---")
        trainer.plot_reconstruction(
            model,
            X_sequences[top_anomaly_idx],
            feature_idx=consumo_feature_idx,
            feature_name=DatasetKeys.CONSUMO_RATIO,
            device='cpu'
        )
    except ValueError:
        print(f"No se encontró la feature '{DatasetKeys.CONSUMO_RATIO}' en las columnas para plotear.")

## 7. Exportación de Resultados para Dashboard

Finalmente, para hacer los resultados accionables y fácilmente explorables, exportamos el dataframe con las anomalías detectadas. El dashboard interactivo del proyecto está diseñado para leer estos resultados directamente.

In [ ]:
# --- 1. Creación de un directorio para el experimento ---
# Para mantener un orden, cada ejecución del notebook que genera resultados
# se guarda en una carpeta de "experimento" única con fecha y hora.
from datetime import datetime

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
experiment_dir = Paths.EXPERIMENTS_DIR / f"exp_{timestamp}"
experiment_dir.mkdir(parents=True, exist_ok=True)

# --- 2. Guardado de los resultados ---
# El dashboard está configurado para leer los resultados desde esta ruta específica.
output_path = experiment_dir / "resultados_completos_tecnicos.csv"
results_df.to_csv(output_path, index=False)

logger.info(f"Resultados del análisis de anomalías guardados en: {output_path}")
print(f"\nResultados exportados y listos para ser visualizados en el Dashboard.")
print("Para iniciar el dashboard, ejecuta en la terminal:")
print("\n./venv/bin/streamlit run dashboard/app.py\n")

## 8. Conclusiones y Próximos Pasos

En este proyecto, hemos desarrollado con éxito un pipeline completo para la detección de anomalías en el consumo de agua utilizando técnicas de Deep Learning. El modelo `LSTM Autoencoder` ha demostrado ser eficaz en aprender los patrones temporales del consumo "normal" y señalar las desviaciones significativas.

**Visualización Interactiva:**
Los resultados de este análisis no son estáticos. Se han exportado y pueden ser explorados de forma interactiva a través del **Dashboard de la aplicación**. Esta herramienta permite a los analistas filtrar por fecha, barrio o uso, y visualizar las anomalías detectadas en un mapa y en gráficos dinámicos, conectando directamente el poder del modelo de IA con la toma de decisiones operativa.

**Próximos Pasos:**

1.  **Refinamiento del Modelo:** Experimentar con arquitecturas como `GRU` o `Transformers` y realizar una búsqueda de hiperparámetros más exhaustiva.

2.  **Ingeniería de Características Avanzada:** Incorporar nuevas fuentes de datos como calendarios de festivos, eventos, o el **dataset de destinos** mencionado.

3.  **Interpretabilidad (XAI):** Integrar `SHAP` para explicar con mayor detalle por qué una lectura es anómala.

4.  **Creación de un Bucle de Retroalimentación:** Implementar un sistema donde los expertos puedan validar las anomalías, permitiendo reentrenar y refinar el modelo periódicamente.